# Modern LLMs: APIs, Evaluation & Practical Applications

In this notebook, we explore **modern large language models** (LLMs) like GPT-4, Claude, and others through their APIs. We'll learn practical skills for working with these models in production, including prompt engineering, few-shot learning, and systematic evaluation.

## Learning Objectives

1. Understand the modern LLM landscape (GPT-4, Claude, LLaMA, etc.)
2. Learn to work with LLM APIs (OpenAI, Anthropic)
3. Master prompt engineering techniques
4. Implement few-shot learning
5. Evaluate LLMs systematically (cost vs performance)
6. Understand when to use fine-tuned BERT vs API-based LLMs

---

## 1. The Modern LLM Landscape

### Major Models & Providers

| Model Family | Provider | Size Range | Key Features | API Access |
|--------------|----------|------------|--------------|------------|
| **GPT-4** | OpenAI | ~1.7T params | Most capable, multimodal | ✅ Paid |
| **GPT-3.5** | OpenAI | 175B params | Fast, cost-effective | ✅ Paid |
| **Claude** | Anthropic | ~100B-200B | Long context (200K), safety-focused | ✅ Paid |
| **Gemini** | Google | Various | Multimodal, integrated with Google services | ✅ Paid |
| **LLaMA** | Meta | 7B-70B | Open weights, self-hostable | ✅ Free (self-host) |
| **Mistral** | Mistral AI | 7B-8x7B (MoE) | Efficient, open weights | ✅ Free/Paid |
| **BERT** | Google | 110M-340M | Fine-tuning required, task-specific | ✅ Free (self-host) |

### Key Capabilities Evolution

```
BERT (2018)          →  GPT-3 (2020)       →  GPT-4 (2023)
────────────────────────────────────────────────────────────
• Fine-tune required    • Zero/few-shot       • Multimodal
• Task-specific         • Prompt-based        • 128K context
• 512 tokens max        • 4K-32K tokens       • Better reasoning
• No generation         • Text generation     • Tool use
• Cheap to run          • Moderate cost       • Higher cost
```

### When to Use What?

**Use Fine-tuned BERT when:**
- You have labeled training data
- Low latency required (<10ms)
- Predictable costs important
- Task is classification/NER/Q&A extraction
- Need to run offline/on-device

**Use API-based LLMs when:**
- No training data available
- Need generation/creative tasks
- Requirements change frequently
- Can tolerate higher latency (100ms-1s)
- Need reasoning/complex instructions

---
## 2. Setup & API Configuration

**⚠️ Important:** This notebook demonstrates API usage patterns. To run these examples, you'll need:
- OpenAI API key (from platform.openai.com)
- Anthropic API key (from console.anthropic.com)

For educational purposes, you can also use mock responses or local models.

In [49]:
# Install required packages
!pip install -q openai anthropic tiktoken pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import time
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For token counting
import tiktoken

# Set your API keys (use environment variables in production!)
#os.environ['OPENAI_API_KEY'] = 'your-key'
#os.environ['ANTHROPIC_API_KEY'] = 'your-key'

# For demonstration, we'll use mock mode
USE_MOCK_RESPONSES = False  # Set to False if you have API keys

print("Setup complete!")
if USE_MOCK_RESPONSES:
    print("⚠️ Running in MOCK mode (no real API calls)")
else:
    print("✅ Running with real API calls")

Setup complete!
✅ Running with real API calls


---
## 3. Working with OpenAI API (GPT-4 / GPT-3.5)

### 3.1 Basic Chat Completion

In [51]:
def call_openai(prompt: str, model: str = "gpt-3.5-turbo", temperature: float = 0.7, max_tokens: int = 150) -> Dict:
    """
    Call OpenAI API with error handling and metrics.

    Args:
        prompt: The user prompt
        model: Model to use (gpt-3.5-turbo, gpt-4, etc.)
        temperature: Sampling temperature
        max_tokens: Maximum tokens to generate

    Returns:
        Dictionary with response, usage stats, and metrics
    """
    if USE_MOCK_RESPONSES:
        # Mock response for demonstration
        time.sleep(0.5)  # Simulate API latency
        return {
            'response': f"[Mock Response] This is a simulated answer to: '{prompt[:50]}...'",
            'model': model,
            'input_tokens': len(prompt.split()) * 1.3,  # Approximate
            'output_tokens': 30,
            'total_tokens': len(prompt.split()) * 1.3 + 30,
            'latency': 0.5,
            'cost': 0.001  # Approximate
        }

    # Real API call
    from openai import OpenAI
    client = OpenAI()

    start_time = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )

    latency = time.time() - start_time

    # Calculate cost (approximate, as of 2024)
    cost_per_1k_input = 0.5 if model == "gpt-3.5-turbo" else 5.0  # USD
    cost_per_1k_output = 1.5 if model == "gpt-3.5-turbo" else 15.0

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = (input_tokens / 1000 * cost_per_1k_input) + (output_tokens / 1000 * cost_per_1k_output)

    return {
        'response': response.choices[0].message.content,
        'model': model,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': response.usage.total_tokens,
        'latency': latency,
        'cost': cost
    }

In [52]:
# Test basic completion
prompt = "Explain what a transformer is in machine learning in 2 sentences."
result = call_openai(prompt, model="gpt-3.5-turbo")

print(f"Prompt: {prompt}\n")
print(f"Response: {result['response']}\n")
print(f"Model: {result['model']}")
print(f"Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total")
print(f"Latency: {result['latency']:.3f}s")
print(f"Cost: ${result['cost']:.6f}")

Prompt: Explain what a transformer is in machine learning in 2 sentences.

Response: A transformer is a model architecture used in machine learning for processing sequential data, such as text or time series. It utilizes self-attention mechanisms to capture long-range dependencies in the data and has been shown to outperform traditional recurrent neural networks in many natural language processing tasks.

Model: gpt-3.5-turbo
Tokens: 21 in + 54 out = 75 total
Latency: 1.688s
Cost: $0.091500


---
## 4. Prompt Engineering

The quality of LLM outputs depends heavily on **how you ask**.

### 4.1 Basic Principles

1. **Be specific**: Clear instructions yield better results
2. **Provide context**: Give background information
3. **Show examples**: Few-shot learning improves accuracy
4. **Specify format**: Tell the model how to structure output
5. **Iterate**: Refine prompts based on results

In [53]:
# Compare bad vs good prompts
prompts_comparison = [
    {
        'name': 'Bad: Vague',
        'prompt': 'Tell me about AI'
    },
    {
        'name': 'Better: Specific',
        'prompt': 'Explain what artificial intelligence is and list 3 main subfields, in 3 sentences.'
    },
    {
        'name': 'Best: Specific + Format',
        'prompt': '''Explain artificial intelligence.

Format your answer as:
1. Definition (1 sentence)
2. Three main subfields (bullet points)
3. One real-world application (1 sentence)'''
    }
]

print("Comparing Prompt Quality:\n")
print("="*80)

for item in prompts_comparison:
    print(f"\n{item['name']}:")
    print(f"Prompt: {item['prompt']}")
    result = call_openai(item['prompt'], max_tokens=200)
    print(f"\nResponse: {result['response']}")
    print("\n" + "="*80)

Comparing Prompt Quality:


Bad: Vague:
Prompt: Tell me about AI

Response: AI, or artificial intelligence, is the simulation of human intelligence in machines that are programmed to think and act like humans. AI technology includes machine learning, natural language processing, computer vision, robotics, and more. AI systems can analyze large amounts of data, learn from patterns, and make decisions without human intervention. AI is used in a wide range of applications, from virtual assistants like Siri and Alexa to self-driving cars and advanced medical diagnosis systems. It has the potential to revolutionize industries, improve efficiency, and solve complex problems. However, there are also concerns about the ethical implications of AI, such as bias in algorithms and job displacement due to automation. Overall, AI is a rapidly evolving field with vast potential for both positive and negative impacts on society.


Better: Specific:
Prompt: Explain what artificial intelligence is and l

### 4.2 Few-Shot Learning

Provide examples in the prompt to teach the model the task pattern.

In [54]:
# Zero-shot vs Few-shot comparison
task = "sentiment classification"

# Zero-shot
zero_shot_prompt = """Classify the sentiment of this text as positive, negative, or neutral.

Text: "This product is amazing!"
Sentiment:"""

# Few-shot
few_shot_prompt = """Classify the sentiment of each text as positive, negative, or neutral.

Text: "I love this product! It exceeded my expectations."
Sentiment: positive

Text: "Terrible quality. Waste of money."
Sentiment: negative

Text: "It's okay, nothing special."
Sentiment: neutral

Text: "This product is amazing!"
Sentiment:"""

print("Zero-shot vs Few-shot Learning:\n")
print("="*80)

print("\nZero-shot:")
result_zero = call_openai(zero_shot_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {result_zero['response']}")

print("\nFew-shot:")
result_few = call_openai(few_shot_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {result_few['response']}")

print("\n" + "="*80)
print("\n✅ Few-shot typically gives more consistent, accurate results")

Zero-shot vs Few-shot Learning:


Zero-shot:
Response: Positive

Few-shot:
Response: positive


✅ Few-shot typically gives more consistent, accurate results


### 4.3 Chain-of-Thought Prompting

Ask the model to "think step by step" for better reasoning.

In [55]:
# Without CoT
direct_prompt = """Q: A store has 15 apples. They sell 7 in the morning and receive 20 more in the afternoon.
Then they sell 12 more. How many apples do they have?

A:"""

# With Chain-of-Thought
cot_prompt = """Q: A store has 15 apples. They sell 7 in the morning and receive 20 more in the afternoon.
Then they sell 12 more. How many apples do they have?

A: Let's solve this step by step:"""

print("Chain-of-Thought Prompting:\n")
print("="*80)

print("\nDirect:")
result_direct = call_openai(direct_prompt, temperature=0.0, max_tokens=50)
print(result_direct['response'])

print("\n" + "="*80)
print("\nChain-of-Thought:")
result_cot = call_openai(cot_prompt, temperature=0.0, max_tokens=100)
print(result_cot['response'])

print("\n" + "="*80)
print("\n✅ CoT helps LLMs show their reasoning and catch errors")

Chain-of-Thought Prompting:


Direct:
They started with 15 apples, sold 7 in the morning, so they had 15 - 7 = 8 apples left.
Then they received 20 more, so they had 8 + 20 = 28 apples.
After selling


Chain-of-Thought:

1. Start with 15 apples.
2. After selling 7 in the morning, they have 15 - 7 = 8 apples left.
3. After receiving 20 more in the afternoon, they have 8 + 20 = 28 apples.
4. After selling 12 more, they have 28 - 12 = 16 apples left.

Therefore, the store has 16 apples in total.


✅ CoT helps LLMs show their reasoning and catch errors


---
## 5. LLM Evaluation Framework

How do we systematically evaluate LLMs?

### 5.1 Task-Specific Evaluation

For tasks with ground truth, we can measure accuracy.

In [56]:
# Create evaluation dataset
eval_dataset = [
    {
        'text': 'I absolutely loved this movie! Best film of the year.',
        'true_sentiment': 'positive'
    },
    {
        'text': 'Terrible service. Never coming back.',
        'true_sentiment': 'negative'
    },
    {
        'text': 'The product works as expected.',
        'true_sentiment': 'neutral'
    },
    {
        'text': 'Waste of time and money. Very disappointed.',
        'true_sentiment': 'negative'
    },
    {
        'text': 'Excellent quality! Highly recommend.',
        'true_sentiment': 'positive'
    }
]

def evaluate_sentiment_model(dataset: List[Dict], model: str = "gpt-3.5-turbo") -> Dict:
    """
    Evaluate LLM on sentiment classification task.
    """
    predictions = []
    total_cost = 0
    total_latency = 0

    for item in dataset:
        prompt = f"""Classify sentiment as: positive, negative, or neutral.

Text: "{item['text']}"
Sentiment:"""

        result = call_openai(prompt, model=model, temperature=0.0, max_tokens=10)
        pred = result['response'].strip().lower()

        # Extract sentiment label
        if 'positive' in pred:
            pred_label = 'positive'
        elif 'negative' in pred:
            pred_label = 'negative'
        else:
            pred_label = 'neutral'

        predictions.append(pred_label)
        total_cost += result['cost']
        total_latency += result['latency']

    # Calculate accuracy
    true_labels = [item['true_sentiment'] for item in dataset]
    accuracy = sum([p == t for p, t in zip(predictions, true_labels)]) / len(dataset)

    return {
        'model': model,
        'accuracy': accuracy,
        'total_cost': total_cost,
        'avg_latency': total_latency / len(dataset),
        'predictions': predictions,
        'true_labels': true_labels
    }

# Evaluate
results = evaluate_sentiment_model(eval_dataset, model="gpt-3.5-turbo")

print("Sentiment Classification Evaluation:\n")
print(f"Model: {results['model']}")
print(f"Accuracy: {results['accuracy']:.2%}")
print(f"Total Cost: ${results['total_cost']:.4f}")
print(f"Avg Latency: {results['avg_latency']:.3f}s")
print("\nPredictions vs True Labels:")
for i, (pred, true) in enumerate(zip(results['predictions'], results['true_labels'])):
    match = "✓" if pred == true else "✗"
    print(f"  {i+1}. {match} Predicted: {pred:8} | True: {true}")

Sentiment Classification Evaluation:

Model: gpt-3.5-turbo
Accuracy: 80.00%
Total Cost: $0.0910
Avg Latency: 0.490s

Predictions vs True Labels:
  1. ✓ Predicted: positive | True: positive
  2. ✓ Predicted: negative | True: negative
  3. ✗ Predicted: positive | True: neutral
  4. ✓ Predicted: negative | True: negative
  5. ✓ Predicted: positive | True: positive


### 5.3 LLM-as-a-Judge

For tasks without clear ground truth (e.g., creative writing, summarization), use another LLM to evaluate.

In [57]:
def llm_judge(text: str, criteria: List[str]) -> Dict[str, float]:
    """
    Use LLM to evaluate text on multiple criteria.

    Args:
        text: Text to evaluate
        criteria: List of evaluation criteria

    Returns:
        Dictionary of scores (1-5) for each criterion
    """
    criteria_str = "\n".join([f"- {c}" for c in criteria])

    prompt = f"""Evaluate the following text on these criteria (rate each 1-5, where 5 is best):

{criteria_str}

Text: "{text}"

Provide your ratings in this format:
Criterion: score
(one per line)"""

    result = call_openai(prompt, temperature=0.0, max_tokens=100)

    # Parse scores (simplified)
    scores = {}
    for criterion in criteria:
        # In mock mode, assign random scores
        if USE_MOCK_RESPONSES:
            scores[criterion] = np.random.randint(3, 6)
        else:
            # Parse from response (you'd need more robust parsing in production)
            scores[criterion] = 4  # Placeholder

    return scores

# Example evaluation
generated_summary = "Machine learning enables computers to learn from data and improve over time without explicit programming."

evaluation_criteria = [
    "Accuracy (factually correct)",
    "Clarity (easy to understand)",
    "Conciseness (not verbose)",
    "Completeness (covers key points)"
]

scores = llm_judge(generated_summary, evaluation_criteria)

print("LLM-as-Judge Evaluation:\n")
print(f"Text: {generated_summary}\n")
print("Scores (1-5):")
for criterion, score in scores.items():
    print(f"  • {criterion}: {score}/5")
print(f"\nAverage Score: {np.mean(list(scores.values())):.2f}/5")

LLM-as-Judge Evaluation:

Text: Machine learning enables computers to learn from data and improve over time without explicit programming.

Scores (1-5):
  • Accuracy (factually correct): 4/5
  • Clarity (easy to understand): 4/5
  • Conciseness (not verbose): 4/5
  • Completeness (covers key points): 4/5

Average Score: 4.00/5


---
## 6. Cost-Performance Optimization

In production, you need to balance quality, cost, and latency.

### 6.1 Cost Projection

In [58]:
# Simulate cost at different scales
def calculate_monthly_cost(queries_per_day: int, cost_per_query: float) -> float:
    return queries_per_day * 30 * cost_per_query

query_volumes = [100, 1000, 10000, 100000]
cost_scenarios = []

for volume in query_volumes:
    cost_scenarios.append({
        'Queries/Day': f"{volume:,}",
        'BERT (fine-tuned)': f"${calculate_monthly_cost(volume, 0.0001):.2f}",
        'GPT-3.5': f"${calculate_monthly_cost(volume, 0.0005):.2f}",
        'GPT-4': f"${calculate_monthly_cost(volume, 0.005):.2f}"
    })

df_cost = pd.DataFrame(cost_scenarios)

print("\n💰 Monthly Cost Projections\n")
print(df_cost.to_string(index=False))
print("\n⚠️ At scale, costs become significant! Optimize wisely.")


💰 Monthly Cost Projections

Queries/Day BERT (fine-tuned)  GPT-3.5     GPT-4
        100             $0.30    $1.50    $15.00
      1,000             $3.00   $15.00   $150.00
     10,000            $30.00  $150.00  $1500.00
    100,000           $300.00 $1500.00 $15000.00

⚠️ At scale, costs become significant! Optimize wisely.


### 6.2 Optimization Strategies

In [59]:
optimization_strategies = pd.DataFrame([
    {
        'Strategy': 'Use cheaper model (GPT-3.5 vs GPT-4)',
        'Cost Reduction': '90%',
        'Quality Impact': 'Moderate',
        'Effort': 'Low'
    },
    {
        'Strategy': 'Reduce output tokens (shorter answers)',
        'Cost Reduction': '30-50%',
        'Quality Impact': 'Low',
        'Effort': 'Low'
    },
    {
        'Strategy': 'Cache frequent queries',
        'Cost Reduction': '50-80%',
        'Quality Impact': 'None',
        'Effort': 'Medium'
    },
    {
        'Strategy': 'Route simple queries to smaller model',
        'Cost Reduction': '40-60%',
        'Quality Impact': 'Low',
        'Effort': 'High'
    },
    {
        'Strategy': 'Fine-tune small model for specific task',
        'Cost Reduction': '95%',
        'Quality Impact': 'Varies',
        'Effort': 'Very High'
    },
    {
        'Strategy': 'Batch processing (delay tolerance)',
        'Cost Reduction': '50%',
        'Quality Impact': 'None',
        'Effort': 'Medium'
    }
])

print("\n🎯 Cost Optimization Strategies\n")
print(optimization_strategies.to_string(index=False))


🎯 Cost Optimization Strategies

                               Strategy Cost Reduction Quality Impact    Effort
   Use cheaper model (GPT-3.5 vs GPT-4)            90%       Moderate       Low
 Reduce output tokens (shorter answers)         30-50%            Low       Low
                 Cache frequent queries         50-80%           None    Medium
  Route simple queries to smaller model         40-60%            Low      High
Fine-tune small model for specific task            95%         Varies Very High
     Batch processing (delay tolerance)            50%           None    Medium
